# Failure Analysis: Copilot Opus 4.5

Analysis of manually-reviewed failure categories across 5 independent runs (101 tasks × 5 runs = 505 result entries).

**Failure Categories** (from `reviewer.py`):
- **Wrong Solution**: Incorrect approach/logic
- **Syntax Error**: AL syntax issues
- **Incorrect File**: Wrong file edited
- **Missing Using**: Missing using statement for namespace
- **Timeout**: Agent timed out and made no changes
- **Other**: Doesn't fit above

In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

RESULT_DIR = Path("result/copilot-opus-4-5")
FAILURE_CATEGORIES = ["Wrong Solution", "Syntax Error", "Incorrect File", "Missing Using", "Timeout", "Other"]

In [2]:
# Load all results from the 5 runs
rows = []
for jsonl_file in sorted(RESULT_DIR.glob("*.jsonl")):
    run_id = jsonl_file.stem
    for line in jsonl_file.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        data = json.loads(line)

        # Extract fields with fallbacks for different naming conventions
        instance_id = data.get("InstanceID") or data.get("instance_id", "unknown")
        scores = data.get("scores", {})
        metrics = data.get("metrics", {}) or {}
        review = data.get("review", {})

        rows.append(
            {
                "run_id": run_id,
                "instance_id": instance_id,
                "build_rate": scores.get("BuildRate", 0),
                "resolution_rate": scores.get("ResolutionRate", 0),
                "resolved": scores.get("ResolutionRate", 0) == 1,
                "builds": scores.get("BuildRate", 0) == 1,
                "failure_category": review.get("failure_category"),
                "duration": metrics.get("duration", 0),
                "tool_calls": metrics.get("tool_calls", 0),
                "total_tokens": metrics.get("total_tokens", 0),
                "turn_count": metrics.get("TurnCount", 0),
                "project": data.get("Project", "Unknown"),
                "output": data.get("output", ""),
            }
        )

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} results from {df['run_id'].nunique()} runs")
print(f"Unique tasks: {df['instance_id'].nunique()}")
df.head()

Loaded 505 results from 5 runs
Unique tasks: 101


,run_id,instance_id,build_rate,resolution_rate,resolved,builds,failure_category,duration,tool_calls,total_tokens,turn_count,project,output
0,21441655230,microsoftInternal__NAV-218062,0,0,False,False,Missing Using,512.775,58,2119600,59,BaseApp,diff --git a/App/Layers/W1/BaseApp/Warehouse/A...
1,21441655230,microsoftInternal__NAV-193649,0,0,False,False,Syntax Error,121.149,31,810900,22,Shopify,diff --git a/App/Apps/W1/Shopify/app/src/Order...
2,21441655230,microsoftInternal__NAV-224447,0,0,False,False,Missing Using,222.267,30,1106100,38,BaseApp,diff --git a/App/Layers/W1/BaseApp/Foundation/...
3,21441655230,microsoftInternal__NAV-178045,0,0,False,False,Timeout,1800.000,0,0,0,BaseApp,
4,21441655230,microsoftInternal__NAV-191624,0,0,False,False,Syntax Error,39.000,8,81000,6,ExcelReports,diff --git a/App/Apps/W1/ExcelReports/app/src/...


## 1. Overview Statistics

In [3]:
# Overall success/failure breakdown
total = len(df)
resolved = df["resolved"].sum()
failed = total - resolved
builds_ok = df["builds"].sum()
build_failed = total - builds_ok

print(f"Total entries: {total}")
print(f"Resolved: {resolved} ({resolved / total * 100:.1f}%)")
print(f"Failed: {failed} ({failed / total * 100:.1f}%)")
print(f"Build succeeded: {builds_ok} ({builds_ok / total * 100:.1f}%)")
print(f"Build failed: {build_failed} ({build_failed / total * 100:.1f}%)")

Total entries: 505
Resolved: 295 (58.4%)
Failed: 210 (41.6%)
Build succeeded: 484 (95.8%)
Build failed: 21 (4.2%)


In [4]:
# Filter to failures only for category analysis
failures_df = df[~df["resolved"]].copy()
print(f"\nAnalyzing {len(failures_df)} failed entries")

# Check review coverage
reviewed = failures_df["failure_category"].notna().sum()
unreviewed = len(failures_df) - reviewed
print(f"Reviewed: {reviewed} ({reviewed / len(failures_df) * 100:.1f}%)")
print(f"Unreviewed: {unreviewed} ({unreviewed / len(failures_df) * 100:.1f}%)")


Analyzing 210 failed entries
Reviewed: 210 (100.0%)
Unreviewed: 0 (0.0%)


## 2. Failure Category Distribution

In [5]:
# Replace NaN with "Unreviewed" for visualization
failures_df["category"] = failures_df["failure_category"].fillna("Unreviewed")

# Count by category
category_counts = failures_df["category"].value_counts().reset_index()
category_counts.columns = ["category", "count"]
category_counts["percentage"] = (category_counts["count"] / len(failures_df) * 100).round(1)

# Define color mapping
color_map = {
    "Wrong Solution": "#EF553B",
    "Syntax Error": "#FFA15A",
    "Incorrect File": "#FECB52",
    "Missing Using": "#00CC96",
    "Timeout": "#636EFA",
    "Other": "#AB63FA",
    "Unreviewed": "#B6E880",
}

fig = px.pie(
    category_counts,
    values="count",
    names="category",
    title="Failure Category Distribution (All Runs)",
    color="category",
    color_discrete_map=color_map,
    hole=0.4,
)
fig.update_traces(textposition="outside", textinfo="percent+label+value")
fig.show()

In [6]:
# Bar chart with build success breakdown
category_build = failures_df.groupby(["category", "builds"]).size().reset_index(name="count")
category_build["build_status"] = category_build["builds"].map({True: "Build OK", False: "Build Failed"})

fig = px.bar(
    category_build,
    x="category",
    y="count",
    color="build_status",
    title="Failure Categories by Build Status",
    barmode="stack",
    color_discrete_map={"Build OK": "#00CC96", "Build Failed": "#EF553B"},
    category_orders={"category": [*FAILURE_CATEGORIES, "Unreviewed"]},
)
fig.update_layout(xaxis_title="Failure Category", yaxis_title="Count")
fig.show()

## 3. Cross-Run Consistency Analysis

Do the same tasks fail the same way across all 5 runs?

In [7]:
# Pivot: instance_id x run_id -> resolution status
resolution_pivot = df.pivot_table(index="instance_id", columns="run_id", values="resolved", aggfunc="first")

# Calculate consistency metrics
n_runs = resolution_pivot.shape[1]
resolution_pivot["pass_count"] = resolution_pivot.sum(axis=1)
resolution_pivot["fail_count"] = n_runs - resolution_pivot["pass_count"]

# Categorize tasks
always_pass = (resolution_pivot["pass_count"] == n_runs).sum()
always_fail = (resolution_pivot["fail_count"] == n_runs).sum()
flaky = n_runs - always_pass - always_fail

print(f"Tasks that ALWAYS pass (5/5 runs): {always_pass}")
print(f"Tasks that ALWAYS fail (0/5 runs): {always_fail}")
print(f"Flaky tasks (pass sometimes): {len(resolution_pivot) - always_pass - always_fail}")

Tasks that ALWAYS pass (5/5 runs): 39
Tasks that ALWAYS fail (0/5 runs): 23
Flaky tasks (pass sometimes): 39


In [8]:
# Visualize pass/fail consistency
pass_distribution = resolution_pivot["pass_count"].value_counts().sort_index().reset_index()
pass_distribution.columns = ["passes_out_of_5", "task_count"]

fig = px.bar(
    pass_distribution,
    x="passes_out_of_5",
    y="task_count",
    title="Task Resolution Consistency Across 5 Runs",
    labels={"passes_out_of_5": "# of Runs Passed", "task_count": "# of Tasks"},
    text="task_count",
    color="passes_out_of_5",
    color_continuous_scale="RdYlGn",
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

In [9]:
# For failures: analyze category consistency
failure_category_pivot = failures_df.pivot_table(index="instance_id", columns="run_id", values="category", aggfunc="first")


# Calculate most common category per task and consistency
def get_category_stats(row):
    categories = row.dropna().tolist()
    if not categories:
        return pd.Series({"most_common": None, "consistency": 0, "n_failures": 0})
    counter = Counter(categories)
    most_common, count = counter.most_common(1)[0]
    return pd.Series(
        {
            "most_common": most_common,
            "consistency": count / len(categories),
            "n_failures": len(categories),
            "unique_categories": len(counter),
        }
    )


category_consistency = failure_category_pivot.apply(get_category_stats, axis=1)
category_consistency = category_consistency[category_consistency["n_failures"] > 0]

print(f"\nCategory Consistency for {len(category_consistency)} tasks that failed at least once:")
print(f"Tasks with 100% consistent failure category: {(category_consistency['consistency'] == 1).sum()}")
print(f"Tasks with mixed failure categories: {(category_consistency['consistency'] < 1).sum()}")
print(f"\nAverage consistency score: {category_consistency['consistency'].mean():.1%}")


Category Consistency for 62 tasks that failed at least once:
Tasks with 100% consistent failure category: 50
Tasks with mixed failure categories: 12

Average consistency score: 93.8%


In [10]:
# Show tasks with inconsistent failure categories (fail differently across runs)
inconsistent = category_consistency[category_consistency["consistency"] < 1].copy()
if len(inconsistent) > 0:
    print(f"\nTasks with inconsistent failure categories ({len(inconsistent)} tasks):")
    inconsistent_details = inconsistent.join(failure_category_pivot)
    display(inconsistent_details.sort_values("consistency"))
else:
    print("All tasks fail consistently with the same category!")


Tasks with inconsistent failure categories (12 tasks):


,most_common,consistency,n_failures,unique_categories,21441655230,21449814776,21469225831,21475222924,21483543210
instance_id,,,,,,,,,
microsoftInternal__NAV-215645,Incorrect File,0.400000,5,3,Incorrect File,Other,Incorrect File,Syntax Error,Other
microsoftInternal__NAV-214557,Wrong Solution,0.500000,4,3,Wrong Solution,NaN,Incorrect File,Missing Using,Wrong Solution
microsoftInternal__NAV-224447,Missing Using,0.500000,2,2,Missing Using,NaN,NaN,NaN,Incorrect File
microsoftInternal__NAV-178045,Timeout,0.666667,3,2,Timeout,NaN,NaN,Timeout,Incorrect File
microsoft__BCApps-5633,Wrong Solution,0.666667,3,2,NaN,Wrong Solution,Wrong Solution,NaN,Missing Using
microsoftInternal__NAV-193649,Missing Using,0.666667,3,2,Syntax Error,Missing Using,Missing Using,NaN,NaN
microsoftInternal__NAV-218062,Missing Using,0.750000,4,2,Missing Using,NaN,Missing Using,Wrong Solution,Missing Using
microsoftInternal__NAV-209450,Incorrect File,0.800000,5,2,Incorrect File,Incorrect File,Incorrect File,Other,Incorrect File
microsoftInternal__NAV-217797,Incorrect File,0.800000,5,2,Incorrect File,Incorrect File,Incorrect File,Wrong Solution,Incorrect File


## 4. Consistently Failing Tasks

Which tasks fail in ALL 5 runs with the same category?

In [11]:
# Tasks that fail all 5 runs with same category
always_fail_tasks = resolution_pivot[resolution_pivot["fail_count"] == n_runs].index.tolist()
consistent_failures = category_consistency[(category_consistency.index.isin(always_fail_tasks)) & (category_consistency["consistency"] == 1)]

print(f"Tasks that fail ALL 5 runs with the SAME category: {len(consistent_failures)}")

# Group by category
if len(consistent_failures) > 0:
    by_category = consistent_failures.groupby("most_common").size().sort_values(ascending=False)
    print("\nBreakdown by failure category:")
    for cat, count in by_category.items():
        print(f"  {cat}: {count} tasks")

Tasks that fail ALL 5 runs with the SAME category: 17

Breakdown by failure category:
  Wrong Solution: 14 tasks
  Syntax Error: 2 tasks
  Incorrect File: 1 tasks


In [12]:
# Detailed table of consistently failing tasks
if len(consistent_failures) > 0:
    # Add project info
    task_projects = df.groupby("instance_id")["project"].first()
    consistent_failures_detail = consistent_failures.copy()
    consistent_failures_detail["project"] = consistent_failures_detail.index.map(task_projects)

    display(consistent_failures_detail[["most_common", "project"]].sort_values(["most_common", "project"]).rename(columns={"most_common": "failure_category"}))

,failure_category,project
instance_id,,
microsoftInternal__NAV-213741,Incorrect File,BaseApp
microsoftInternal__NAV-183399,Syntax Error,BaseApp
microsoftInternal__NAV-191624,Syntax Error,ExcelReports
microsoftInternal__NAV-176426,Wrong Solution,BaseApp
microsoftInternal__NAV-177750,Wrong Solution,BaseApp
microsoftInternal__NAV-181900,Wrong Solution,BaseApp
microsoftInternal__NAV-182354,Wrong Solution,BaseApp
microsoftInternal__NAV-204450,Wrong Solution,BaseApp
microsoftInternal__NAV-206135,Wrong Solution,BaseApp


In [13]:
# Heatmap: Failure Category x Project
if len(consistent_failures) > 0:
    consistent_failures_detail = consistent_failures.copy()
    consistent_failures_detail["project"] = consistent_failures_detail.index.map(task_projects)

    heatmap_data = consistent_failures_detail.groupby(["most_common", "project"]).size().unstack(fill_value=0)

    fig = px.imshow(
        heatmap_data,
        title="Consistent Failures: Category x Project",
        labels={"x": "Project", "y": "Failure Category", "color": "Count"},
        aspect="auto",
        color_continuous_scale="Reds",
        text_auto=True,
    )
    fig.show()